In [24]:
import os
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats, point_query
import pandas as pd
from glob import glob
import numpy as np

In [ ]:
# === Read wind map (GeoTIFF) ===
tc_path = '../data/tc/tc_yagi.tif'
wind_src = rasterio.open(tc_path)

# === Read flood map (Shapefile) ===
flood_path = '../data/flood/AIT-VAP001-VNM.shp'
# all_shps = glob(os.path.join(flood_path, "*.shp"))
# flood_gdf = [gpd.read_file(shp).to_crs("EPSG:32648") for shp in all_shps]
flood_gdf = gpd.read_file(flood_path)
flood_gdf = flood_gdf.to_crs(wind_src.crs)

# === Read power network data ===
lines_gdf = gpd.read_file('../outputs/table_lines_200m_update.gpkg').to_crs(wind_src.crs)
nodes_gdf = gpd.read_file('../outputs/table_nodes_200m_update.gpkg').to_crs(wind_src.crs)
gens_gdf = gpd.read_file('../outputs/plant_update.gpkg').to_crs(wind_src.crs)
loads_gdf = gpd.read_file('../../cascading_failure/mapping_industries_economic_sectors/outputs/landuse_sites_gdf_add_bus.gpkg').to_crs(wind_src.crs)

In [16]:
def extract_wind_speed(gdf, raster_path):
    results = []
    with rasterio.open(raster_path) as src:
        for geom in gdf.geometry:
            if geom.is_empty:
                results.append(None)
                continue
            if geom.geom_type == 'Point':
                val = point_query(geom, raster_path, interpolate='nearest')
                results.append(val[0] if val else None)
            else:
                # 为线几何创建 10m buffer，确保能覆盖 raster 像素
                geom_proc = geom.buffer(10) if geom.geom_type.startswith('Line') else geom
                stats = zonal_stats(
                    geom_proc,
                    src.read(1),
                    affine=src.transform,
                    stats=['max'],
                    nodata=src.nodata,
                    all_touched=True
                )
                results.append(stats[0]['max'] if stats and stats[0]['max'] is not None else None)
    
    gdf['wind_speed'] = results
    gdf['wind_speed'] = gdf['wind_speed'].fillna(0)

    return gdf

In [18]:
def extract_flood_depth(elements_gdf, flood_gdf, depth_col="Field"):
    elements_gdf["flood_depth"] = None

    # 使用空间连接，找出所有有接触的组合
    joined = gpd.sjoin(elements_gdf, flood_gdf, how="left", predicate="intersects")
    
    # 聚合（取最大值），避免一个元素与多个 flood 区域交叠时丢信息
    grouped = joined.groupby(joined.index).agg({depth_col: "max"})
    
    # 把结果写回原始 dataframe
    elements_gdf.loc[grouped.index, "flood_depth"] = grouped[depth_col]

    elements_gdf["flood_depth"] = elements_gdf["flood_depth"].fillna(0)

    return elements_gdf

In [ ]:
lines_gdf = extract_wind_speed(lines_gdf, tc_path)
lines_gdf = extract_flood_depth(lines_gdf, flood_gdf)
lines_gdf.head()

C:\Users\mye500\AppData\Local\Temp\ipykernel_18632\968571012.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  elements_gdf["flood_depth"] = elements_gdf["flood_depth"].fillna(0)


In [ ]:
"""
    TODO:
        Read sheets (corresponding to different hazard_type: 'wind' or 'flood'）
        dam_class: element types, i.e., plant (specific type, e.g., wind turbine), substation, tower, line, building (for loads)
        Intergrate 'load_curves' function into 'monte_carlo_simulation' function, 
        provide values of parameters 'hazard_type' and 'dam_class' to read specific fragility curve.
    
    TODO: Prepare excel file for fragility curves
"""
# === Read fragility curves ===
def load_curves(curve_path, sheet_name):
    curves = pd.read_excel(curve_path, sheet_name=sheet_name, skiprows=5)
    curves.set_index(curves.columns[0])
    curves.columns = ["Wind speed", "Tower", "Line"]

    curves = curves.apply(pd.to_numeric, errors='coerce')

    # # design wind speed (dws)
    # dws = 44
    # # shift design wind speed of all curves to 60 m/s
    # scaling_factor = dws / 60
    # curves = curves.apply(lambda x: x * scaling_factor if pd.api.types.is_numeric_dtype(x) else x)

    #interpolate the curves to fill missing values
    # curves = curves.interpolate()
    return curves

curve_path = '../data/fragility_curves.xlsx'
curves = load_curves(curve_path, sheet_name='test')

curves.head()

,Wind speed,Tower,Line
0,0,0.000000e+00,0.000000e+00
1,1,0.000000e+00,5.150900e-10
2,2,3.549661e-278,1.858409e-08
3,3,4.069272e-221,1.513837e-07
4,4,1.242419e-184,6.705013e-07


In [ ]:
from scipy.interpolate import interp1d

# 蒙特卡洛模拟函数
def monte_carlo_simulation(gdf, curves, hazard_type, dam_class, num_iterations=10):
    # 复制原始数据，初始化失败列
    simulation_result = gdf.copy()
    
    # 根据hazard type和dam_class获取 fragility 曲线
    if hazard_type == 'wind':
        fail_prob_curve = curves[["Wind speed", dam_class]]
        interp_func = interp1d(fail_prob_curve["Wind speed"], 
                            fail_prob_curve[dam_class], 
                            kind='linear', fill_value='extrapolate')

        for iteration in range(num_iterations):
            fail_column = f'fail_iter_{iteration+1}'
            failures = []

            for i in range(len(simulation_result)):
                wind_speed = simulation_result.loc[i, 'wind_speed']
                fail_prob = interp_func(wind_speed)
                rand_num = np.random.uniform(0, 1)
                failures.append(1 if rand_num <= fail_prob else 0)
            
            simulation_result[fail_column] = failures
    
    elif hazard_type == 'flood':
        fail_prob_curve = curves[["Depth", dam_class]]
        interp_func = interp1d(fail_prob_curve["Depth"], 
                            fail_prob_curve[dam_class], 
                            kind='linear', fill_value='extrapolate')

        for iteration in range(num_iterations):
            fail_column = f'fail_iter_{iteration+1}'
            failures = []

            for i in range(len(simulation_result)):
                flood_depth = simulation_result.loc[i, 'flood_depth']
                fail_prob = interp_func(flood_depth)
                rand_num = np.random.uniform(0, 1)
                failures.append(1 if rand_num <= fail_prob else 0)
            
            simulation_result[fail_column] = failures        

    return simulation_result

simulation_results = monte_carlo_simulation(lines_gdf, curves, 'wind', 'Line', num_iterations=10)
simulation_results.head()

,Country,osm_id,fromNode,toNode,voltage,Length,R,XL,XC,Itherm,...,fail_iter_1,fail_iter_2,fail_iter_3,fail_iter_4,fail_iter_5,fail_iter_6,fail_iter_7,fail_iter_8,fail_iter_9,fail_iter_10
0,VN,153608701,NODEVN3270a,NODEVN0001,110.0,2.692679,,,,,...,0,1,1,1,1,0,0,1,1,1
1,VN,153608701,NODEVN3270a,NODEVN0001,110.0,2.692679,,,,,...,1,1,1,1,0,1,0,1,1,0
2,VN,176708144,NODEVN0002,NODEVN3650,110.0,27.367648,,,,,...,1,0,0,0,1,0,1,1,1,1
3,VN,241194661,NODEVN0003a,NODEVN3273b,220.0,16.135774,,,,,...,1,0,1,0,1,1,1,1,0,0
4,VN,241194661,NODEVN0003a,NODEVN3273b,220.0,16.135774,,,,,...,1,1,0,1,1,1,1,0,1,1
